In [ ]:
import pandas as pd

# 1. Cargar el dataset de Airbnb
ruta_datos = "data/raw/airbnb/insideairbnb_barcelona_2026-06-24_listings.csv"
df = pd.read_csv(ruta_datos, low_memory=False)

# Total de registros
total_registros = len(df)
print(f"Total de registros cargados: {total_registros:,}")

In [ ]:
# 2. Clasificación de licencias según tipo de registro (HUTB/TU, NT, HB, AJ y nulas/sin número)

def clasificar_licencia(row):
    """
    Clasifica el registro según el tipo de licencia utilizando las reglas de negocio:
    - HUTB/HUT o código TU (ej. ESFCTU): Apartamento turístico (HUTB / TU).
    - Código NT (ej. ESFCNT): No turístico (Alquiler de temporada / NT).
    - Código HB: Licencia hotelera (HB).
    - Código AJ: Licencia albergue (AJ).
    - Texto de exención sin número de registro real o campo nulo: Sin licencia / Nula.
    - Otros códigos registrados.
    
    Args:
        row (pd.Series): Fila del DataFrame con el campo 'license'.
        
    Returns:
        str: Categoría asignada al tipo de licencia.
    """
    lic_val = row["license"]
    if pd.isna(lic_val) or str(lic_val).strip() in ["", "nan"]:
        return "Sin licencia / Nula"
    
    lic = str(lic_val)
    lic_upper = lic.upper()
    has_digits = any(c.isdigit() for c in lic)
    
    # 1. Códigos turísticos: HUTB/HUT o código nacional TU (ej. ESFCTU..., ESHFTU...)
    if "HUTB" in lic_upper or "HUT" in lic_upper or "ESFCTU" in lic_upper or "ESHFTU" in lic_upper:
        return "Apartamento turístico (HUTB / TU)"
        
    # 2. Códigos no turísticos: NT (ej. ESFCNT..., ESHFNT...)
    if "ESFCNT" in lic_upper or "ESHFNT" in lic_upper or ("NT" in lic_upper and has_digits):
        return "No turístico (Alquiler de temporada / NT)"
        
    # 3. Licencia hotelera (HB)
    if "HB-" in lic_upper or ("HB" in lic_upper and has_digits):
        return "Licencia hotelera (HB)"
        
    # 4. Licencia albergue (AJ)
    if "AJ" in lic_upper and has_digits:
        return "Licencia albergue (AJ)"
        
    # 5. Texto plano de exención sin número de registro real (ej. 'Exempt - seasonal rental')
    if not has_digits or "EXEMPT" in lic_upper:
        return "Sin licencia / Nula"
        
    return "Otras licencias registradas"

df["categoria_licencia"] = df.apply(clasificar_licencia, axis=1)

# Tabla de resumen general con Total y Porcentaje (%)
resumen = df["categoria_licencia"].value_counts().reset_index()
resumen.columns = ["Categoría de Licencia", "Total Registros"]
resumen["Porcentaje (%)"] = (resumen["Total Registros"] / total_registros * 100).round(2)

print("--- DESGLOSE TOTAL DE LICENCIAS Y PORCENTAJE (%) ---")
display(resumen)

In [ ]:
# 3. Análisis cruzado por categoría de licencia y tipo de habitación (room_type)

def analizar_licencias_por_tipo_habitacion(df_datos):
    """
    Analiza el desglose de categorías de licencia cruzado exclusivamente con el tipo
    de habitación ('room_type') y muestra los textos de licencia más habituales.
    
    Args:
        df_datos (pd.DataFrame): DataFrame con la columna 'categoria_licencia' y 'room_type'.
        
    Returns:
        pd.DataFrame: Tabla cruzada de categorías de licencia por room_type.
    """
    print("--- 1. Valores más frecuentes en la columna 'license' ---")
    display(df_datos["license"].value_counts(dropna=False).head(15).reset_index())
    
    print("\n--- 2. Desglose por categoría de licencia y tipo de habitación ('room_type') ---")
    desglose = pd.crosstab(
        df_datos["categoria_licencia"], 
        df_datos["room_type"], 
        margins=True, 
        margins_name="Total"
    ).sort_values(by="Total", ascending=False)
    
    display(desglose)
    
    return desglose

# Ejecutar análisis cruzado
tabla_desglose = analizar_licencias_por_tipo_habitacion(df)


In [ ]:
# 4. Filtrado de apartamentos turísticos y sin licencia sin comentarios en los últimos 12 meses

def analizar_inactivos_12m(df_datos):
    """
    Filtra las categorías 'Apartamento turístico (HUTB / TU)' y 'Sin licencia / Nula',
    excluyendo los registros que tengan comentarios en los últimos 12 meses
    (number_of_reviews_ltm > 0).
    
    Calcula los totales y porcentajes combinados ('de ambos') y por separado.
    
    Args:
        df_datos (pd.DataFrame): DataFrame original con 'categoria_licencia' y 'number_of_reviews_ltm'.
        
    Returns:
        pd.DataFrame: DataFrame filtrado con los registros sin comentarios en 12m.
    """
    # Categorías de interés
    categorias_interes = ["Apartamento turístico (HUTB / TU)", "Sin licencia / Nula"]
    
    # Filtrar solo esas 2 categorías
    df_interes = df_datos[df_datos["categoria_licencia"].isin(categorias_interes)].copy()
    
    # Quitar registros con comentarios en los últimos 12 meses (conservar number_of_reviews_ltm == 0)
    df_sin_resenas = df_interes[df_interes["number_of_reviews_ltm"] == 0].copy()
    
    total_general = len(df_datos)
    total_interes = len(df_interes)
    total_sin_resenas_ambos = len(df_sin_resenas)
    
    print("--- 1. RESUMEN COMBINADO (AMBOS GRUPOS SIN RESEÑAS EN 12 MESES) ---")
    print(f"Total combinado sin comentarios en 12m: {total_sin_resenas_ambos:,}")
    print(f"Porcentaje sobre las 2 categorías ({total_interes:,}): {(total_sin_resenas_ambos / total_interes * 100):.2f}%")
    print(f"Porcentaje sobre el total del dataset ({total_general:,}): {(total_sin_resenas_ambos / total_general * 100):.2f}%\n")
    
    # Desglose por separado
    registros_separado = []
    for cat in categorias_interes:
        tot_cat_original = len(df_datos[df_datos["categoria_licencia"] == cat])
        tot_cat_sin_resenas = len(df_sin_resenas[df_sin_resenas["categoria_licencia"] == cat])
        pct_sobre_categoria = (tot_cat_sin_resenas / tot_cat_original * 100) if tot_cat_original > 0 else 0
        pct_sobre_dataset = (tot_cat_sin_resenas / total_general * 100)
        
        registros_separado.append({
            "Categoría": cat,
            "Sin comentarios (12m)": tot_cat_sin_resenas,
            "Total categoría": tot_cat_original,
            "% sobre su categoría": round(pct_sobre_categoria, 2),
            "% sobre total dataset": round(pct_sobre_dataset, 2)
        })
        
    tabla_separado = pd.DataFrame(registros_separado)
    print("--- 2. DESGLOSE POR SEPARADO ---")
    display(tabla_separado)
    
    return df_sin_resenas

# Ejecutar análisis de inactivos en los últimos 12 meses
df_inactivos_12m = analizar_inactivos_12m(df)


In [ ]:
# 5. Detección de duplicados literales en alojamientos activos (con comentarios en los últimos 12 meses)

def analizar_duplicados_activos_12m(df_datos):
    """
    Filtra los alojamientos de las categorías 'Apartamento turístico (HUTB / TU)' y
    'Sin licencia / Nula' que hayan estado activos en los últimos 12 meses (number_of_reviews_ltm > 0).
    
    Identifica duplicados literales bajo diferentes criterios (identidad exacta, mismo nombre/anfitrión,
    y misma licencia), mostrando los totales y porcentajes agrupados (juntos) y por separado.
    
    Args:
        df_datos (pd.DataFrame): DataFrame original con las categorías de licencia y número de reseñas.
        
    Returns:
        pd.DataFrame: DataFrame filtrado con el universo activo analizado.
    """
    categorias_interes = ["Apartamento turístico (HUTB / TU)", "Sin licencia / Nula"]
    
    # 1. Filtrar universo activo (con reseñas en los últimos 12 meses)
    mask_activos = df_datos["categoria_licencia"].isin(categorias_interes) & (df_datos["number_of_reviews_ltm"] > 0)
    df_activos = df_datos[mask_activos].copy()
    
    tot_general_activos = len(df_activos)
    tot_apt_activos = len(df_activos[df_activos["categoria_licencia"] == "Apartamento turístico (HUTB / TU)"])
    tot_sin_activos = len(df_activos[df_activos["categoria_licencia"] == "Sin licencia / Nula"])
    
    print("--- 1. UNIVERSO ACTIVO (CON COMENTARIOS EN ÚLTIMOS 12 MESES) ---")
    print(f"Total activo combinado (ambas categorías): {tot_general_activos:,}")
    print(f"  - Apartamentos turísticos (HUTB / TU): {tot_apt_activos:,}")
    print(f"  - Sin licencia / Nula: {tot_sin_activos:,}\n")
    
    # 2. Criterios de duplicación
    criterios = {
        "Duplicados exactos por identidad (nombre, anfitrión, coords, tipo, precio)": ["name", "host_id", "latitude", "longitude", "room_type", "price"],
        "Duplicados por mismo nombre y anfitrión (name, host_id)": ["name", "host_id"],
        "Duplicados por misma licencia (código repetido)": ["license"]
    }
    
    resultados_duplicados = []
    
    for nombre_criterio, columnas_criterio in criterios.items():
        if columnas_criterio == ["license"]:
            mask_dup = df_activos["license"].notna() & df_activos.duplicated(subset=columnas_criterio, keep=False)
        else:
            mask_dup = df_activos.duplicated(subset=columnas_criterio, keep=False)
            
        df_dup = df_activos[mask_dup]
        
        n_juntos = len(df_dup)
        pct_juntos = (n_juntos / tot_general_activos * 100) if tot_general_activos > 0 else 0
        
        n_apt = len(df_dup[df_dup["categoria_licencia"] == "Apartamento turístico (HUTB / TU)"])
        pct_apt = (n_apt / tot_apt_activos * 100) if tot_apt_activos > 0 else 0
        
        n_sin = len(df_dup[df_dup["categoria_licencia"] == "Sin licencia / Nula"])
        pct_sin = (n_sin / tot_sin_activos * 100) if tot_sin_activos > 0 else 0
        
        resultados_duplicados.append({
            "Criterio de Duplicación": nombre_criterio,
            "Total Juntos": n_juntos,
            "% Juntos": round(pct_juntos, 2),
            "Apt. Turístico": n_apt,
            "% Apt. Turístico": round(pct_apt, 2),
            "Sin Licencia": n_sin,
            "% Sin Licencia": round(pct_sin, 2)
        })
        
    tabla_duplicados = pd.DataFrame(resultados_duplicados)
    
    print("--- 2. ANÁLISIS DE DUPLICADOS LITERALES (TOTALES Y % JUNTOS Y POR SEPARADO) ---")
    display(tabla_duplicados)
    
    return df_activos

# Ejecutar análisis de duplicados en el universo activo
df_universo_activo = analizar_duplicados_activos_12m(df)


In [ ]:
# 6. Desglose de duplicados por combinaciones clave (Nombre+Anfitrión, Licencia, Anfitrión+Licencia, y Anfitrión+Licencia+Precio)

def desglosar_duplicados_por_tipo(df_datos):
    """
    Analiza los alojamientos duplicados según diversas combinaciones de campos:
    - Nombre + Anfitrión (name, host_id).
    - Misma Licencia (license).
    - Nombre + Anfitrión + Licencia (coincidencia triple).
    - Anfitrión + Licencia (host_id, license).
    - Anfitrión + Licencia + Precio (host_id, license, price).
    
    Muestra dos tablas separadas desglosadas por 'room_type':
    - Tabla 1: Apartamentos turísticos (HUTB / TU).
    - Tabla 2: Sin licencia / Nula.
    
    Args:
        df_datos (pd.DataFrame): DataFrame original de Airbnb.
        
    Returns:
        tuple: (pd.DataFrame tabla_apt_turisticos, pd.DataFrame tabla_sin_licencia)
    """
    categorias_interes = ["Apartamento turístico (HUTB / TU)", "Sin licencia / Nula"]
    
    # Universo activo (con reseñas en los últimos 12 meses)
    mask_activos = df_datos["categoria_licencia"].isin(categorias_interes) & (df_datos["number_of_reviews_ltm"] > 0)
    df_activos = df_datos[mask_activos].copy()
    
    # Criterio 1: Mismo nombre y anfitrión (name, host_id)
    df_activos["dup_nombre_anfitrion"] = df_activos.duplicated(subset=["name", "host_id"], keep=False)
    
    # Criterio 2: Misma licencia (license, excluyendo nulos)
    df_activos["dup_misma_licencia"] = df_activos["license"].notna() & df_activos.duplicated(subset=["license"], keep=False)
    
    # Criterio 3: Coincidencia Triple (name, host_id, license)
    df_activos["dup_triple"] = df_activos["license"].notna() & df_activos.duplicated(subset=["name", "host_id", "license"], keep=False)
    
    # Criterio 4: Anfitrión + Licencia (host_id, license)
    df_activos["dup_host_lic"] = df_activos["license"].notna() & df_activos.duplicated(subset=["host_id", "license"], keep=False)
    
    # Criterio 5: Anfitrión + Licencia + Precio (host_id, license, price)
    df_activos["dup_host_lic_price"] = df_activos["license"].notna() & df_activos.duplicated(subset=["host_id", "license", "price"], keep=False)
    
    col_agrupacion = ["property_type", "room_type"] if "property_type" in df_datos.columns else ["room_type"]
    
    tablas_resultado = {}
    
    for cat in categorias_interes:
        df_cat = df_activos[df_activos["categoria_licencia"] == cat]
        
        # Agrupar recuentos por criterio
        counts_nh = df_cat[df_cat["dup_nombre_anfitrion"]].groupby(col_agrupacion).size()
        counts_lic = df_cat[df_cat["dup_misma_licencia"]].groupby(col_agrupacion).size()
        counts_triple = df_cat[df_cat["dup_triple"]].groupby(col_agrupacion).size()
        counts_hl = df_cat[df_cat["dup_host_lic"]].groupby(col_agrupacion).size()
        counts_hlp = df_cat[df_cat["dup_host_lic_price"]].groupby(col_agrupacion).size()
        
        tipos_unicos = df_cat[col_agrupacion].drop_duplicates().sort_values(by=col_agrupacion)
        tabla = tipos_unicos.copy()
        
        if len(col_agrupacion) == 1:
            col_key = col_agrupacion[0]
            tabla["Duplicados Mismo Nombre + Anfitrión"] = tabla[col_key].map(counts_nh).fillna(0).astype(int)
            tabla["Duplicados Misma Licencia"] = tabla[col_key].map(counts_lic).fillna(0).astype(int)
            tabla["Duplicados Mismo Nombre + Anfitrión + Licencia"] = tabla[col_key].map(counts_triple).fillna(0).astype(int)
            tabla["Duplicados Anfitrión + Licencia"] = tabla[col_key].map(counts_hl).fillna(0).astype(int)
            tabla["Duplicados Anfitrión + Licencia + Precio"] = tabla[col_key].map(counts_hlp).fillna(0).astype(int)
        else:
            tabla["Duplicados Mismo Nombre + Anfitrión"] = tabla.set_index(col_agrupacion).index.map(counts_nh).fillna(0).astype(int)
            tabla["Duplicados Misma Licencia"] = tabla.set_index(col_agrupacion).index.map(counts_lic).fillna(0).astype(int)
            tabla["Duplicados Mismo Nombre + Anfitrión + Licencia"] = tabla.set_index(col_agrupacion).index.map(counts_triple).fillna(0).astype(int)
            tabla["Duplicados Anfitrión + Licencia"] = tabla.set_index(col_agrupacion).index.map(counts_hl).fillna(0).astype(int)
            tabla["Duplicados Anfitrión + Licencia + Precio"] = tabla.set_index(col_agrupacion).index.map(counts_hlp).fillna(0).astype(int)
            
        tablas_resultado[cat] = tabla
        
    print("================ TABLA 1: APARTAMENTOS TURÍSTICOS (HUTB / TU) ================")
    display(tablas_resultado["Apartamento turístico (HUTB / TU)"])
    
    print("\n================ TABLA 2: SIN LICENCIA / NULA ================")
    display(tablas_resultado["Sin licencia / Nula"])
    
    return tablas_resultado["Apartamento turístico (HUTB / TU)"], tablas_resultado["Sin licencia / Nula"]

# Ejecutar desglose por tipo de habitación y propiedad en tablas separadas
tabla_apt_dup, tabla_sin_dup = desglosar_duplicados_por_tipo(df)


In [ ]:
# 7. Tabla 1: Duplicados por Mismo Nombre, Mismo Anfitrión, Misma Licencia y coincidencia de los tres (Apartamentos turísticos)

def analizar_duplicados_exactos_turistico(df_datos):
    """
    Muestra para 'Entire home/apt' y 'Private room' los anuncios duplicados por:
    - Mismo Nombre (títulos iguales entre cualquier anuncio).
    - Mismo Anfitrión (anuncios duplicados publicados por el mismo host: name + host_id).
    - Misma Licencia (código de licencia repetido).
    - Coincidencia Triple (mismo nombre, anfitrión y licencia a la vez).
    
    Args:
        df_datos (pd.DataFrame): DataFrame original de Airbnb.
        
    Returns:
        pd.DataFrame: Tabla 1 corregida centrada en anuncios duplicados.
    """
    mask_activos = (df_datos["categoria_licencia"] == "Apartamento turístico (HUTB / TU)") & (df_datos["number_of_reviews_ltm"] > 0)
    df_apt = df_datos[mask_activos].copy()
    
    filas_t1 = []
    for rtype in ["Entire home/apt", "Private room"]:
        sub = df_apt[df_apt["room_type"] == rtype].copy()
        filas_t1.append({
            "Tipo de Habitación": rtype,
            "Duplicados Mismo Nombre": sub.duplicated(subset=["name"], keep=False).sum(),
            "Duplicados Mismo Anfitrión (Nombre + Host)": sub.duplicated(subset=["name", "host_id"], keep=False).sum(),
            "Duplicados Misma Licencia": (sub["license"].notna() & sub.duplicated(subset=["license"], keep=False)).sum(),
            "Coincidencia Triple (Nombre + Host + Licencia)": (sub["license"].notna() & sub.duplicated(subset=["name", "host_id", "license"], keep=False)).sum()
        })
        
    tabla_1 = pd.DataFrame(filas_t1)
    print("================ TABLA 1: APARTAMENTOS TURÍSTICOS (HUTB / TU) ================")
    display(tabla_1)
    return tabla_1

# Ejecutar Tabla 1 corregida
tabla_1_resultado = analizar_duplicados_exactos_turistico(df)


In [ ]:
# 8. Tabla 2: Hosts con un mismo anuncio repetido (Alojamientos sin licencia / nula)

def analizar_hosts_mismo_anuncio(df_datos):
    """
    Muestra para 'Private room' y 'Entire home/apt' en alojamientos sin licencia cuántos hosts
    tienen un mismo anuncio exactamente repetido (mismo nombre y anfitrión) y cuántos anuncios suman.
    
    Args:
        df_datos (pd.DataFrame): DataFrame original de Airbnb.
        
    Returns:
        pd.DataFrame: Tabla 2 con la relación de hosts y anuncios repetidos.
    """
    mask_activos = (df_datos["categoria_licencia"] == "Sin licencia / Nula") & (df_datos["number_of_reviews_ltm"] > 0)
    df_sin = df_datos[mask_activos].copy()
    
    filas_t2 = []
    for rtype in ["Private room", "Entire home/apt"]:
        sub = df_sin[df_sin["room_type"] == rtype].copy()
        dup_name_host = sub.duplicated(subset=["name", "host_id"], keep=False)
        hosts_count = sub[dup_name_host]["host_id"].nunique()
        anuncios_count = dup_name_host.sum()
        
        filas_t2.append({
            "Tipo de Habitación": rtype,
            "Hosts con Mismo Anuncio": hosts_count,
            "Anuncios Repetidos (Mismo Nombre + Host)": anuncios_count
        })
        
    tabla_2 = pd.DataFrame(filas_t2)
    print("================ TABLA 2: SIN LICENCIA / NULA ================")
    display(tabla_2)
    return tabla_2

# Ejecutar Tabla 2
tabla_2_resultado = analizar_hosts_mismo_anuncio(df)


In [ ]:
# 9. Desduplicación progresiva acumulada (incluyendo eliminación en Sin Licencia por Mismo Precio+Nombre+Anfitrión)

def aplicar_desduplicacion_progresiva(df_datos):
    """
    Toma los alojs. activos de 'Apartamento turístico (HUTB / TU)' y 'Sin licencia / Nula'.
    1. Conserva solo 1 registro para duplicados en 'Anfitrión + Licencia + Precio'.
    2. Conserva solo 1 registro para duplicados en 'Nombre + Anfitrión + Licencia'.
    3. Conserva solo 1 registro para duplicados en 'Anfitrión + Licencia'.
    4. Conserva solo 1 registro para duplicados en Sin Licencia con 'Precio + Nombre + Anfitrión'.
    5. Recalcula las tablas para observar el impacto en los conteos de duplicados.
    
    Args:
        df_datos (pd.DataFrame): DataFrame original de Airbnb.
        
    Returns:
        tuple: (pd.DataFrame df_desduplicado, dict tablas_resultantes)
    """
    categorias_interes = ["Apartamento turístico (HUTB / TU)", "Sin licencia / Nula"]
    mask_activos = df_datos["categoria_licencia"].isin(categorias_interes) & (df_datos["number_of_reviews_ltm"] > 0)
    df_activos = df_datos[mask_activos].copy()
    
    tot_inicial = len(df_activos)
    
    # 1. Desduplicar 'Anfitrión + Licencia + Precio'
    mask_lic_1 = df_activos["license"].notna()
    df_lic_1 = df_activos[mask_lic_1].drop_duplicates(subset=["host_id", "license", "price"], keep="first")
    df_paso_1 = pd.concat([df_lic_1, df_activos[~mask_lic_1]], ignore_index=True)
    
    # 2. Desduplicar 'Nombre + Anfitrión + Licencia'
    mask_lic_2 = df_paso_1["license"].notna()
    df_lic_2 = df_paso_1[mask_lic_2].drop_duplicates(subset=["name", "host_id", "license"], keep="first")
    df_paso_2 = pd.concat([df_lic_2, df_paso_1[~mask_lic_2]], ignore_index=True)
    
    # 3. Desduplicar 'Anfitrión + Licencia'
    mask_lic_3 = df_paso_2["license"].notna()
    df_lic_3 = df_paso_2[mask_lic_3].drop_duplicates(subset=["host_id", "license"], keep="first")
    df_paso_3 = pd.concat([df_lic_3, df_paso_2[~mask_lic_3]], ignore_index=True)
    
    # 4. Desduplicar Sin Licencia por 'Precio + Nombre + Anfitrión'
    mask_sin = df_paso_3["categoria_licencia"] == "Sin licencia / Nula"
    df_sin_dedup = df_paso_3[mask_sin].drop_duplicates(subset=["price", "name", "host_id"], keep="first")
    df_dedup = pd.concat([df_paso_3[~mask_sin], df_sin_dedup], ignore_index=True)
    
    tot_final = len(df_dedup)
    eliminados = tot_inicial - tot_final
    
    print(f"--- PROCESO DE DESDUPLICACIÓN ACUMULADO --- ")
    print(f"Registros activos iniciales: {tot_inicial:,}")
    print(f"Registros activos tras desduplicar Sin Licencia por Precio+Nombre+Host: {tot_final:,} (se eliminaron {eliminados:,} duplicados sobrantes en total)\n")
    
    # Recalcular banderas de duplicidad
    df_dedup["dup_nombre_host"] = df_dedup.duplicated(subset=["name", "host_id"], keep=False)
    df_dedup["dup_licencia"] = df_dedup["license"].notna() & df_dedup.duplicated(subset=["license"], keep=False)
    df_dedup["dup_triple"] = df_dedup["license"].notna() & df_dedup.duplicated(subset=["name", "host_id", "license"], keep=False)
    df_dedup["dup_host_lic"] = df_dedup["license"].notna() & df_dedup.duplicated(subset=["host_id", "license"], keep=False)
    df_dedup["dup_host_lic_price"] = df_dedup["license"].notna() & df_dedup.duplicated(subset=["host_id", "license", "price"], keep=False)
    df_dedup["dup_precio_nombre_host"] = df_dedup.duplicated(subset=["price", "name", "host_id"], keep=False)
    
    col_agrupacion = ["property_type", "room_type"] if "property_type" in df_datos.columns else ["room_type"]
    tablas_resultado = {}
    
    for cat in categorias_interes:
        df_cat = df_dedup[df_dedup["categoria_licencia"] == cat]
        
        counts_nh = df_cat[df_cat["dup_nombre_host"]].groupby(col_agrupacion).size()
        counts_lic = df_cat[df_cat["dup_licencia"]].groupby(col_agrupacion).size()
        counts_triple = df_cat[df_cat["dup_triple"]].groupby(col_agrupacion).size()
        counts_hl = df_cat[df_cat["dup_host_lic"]].groupby(col_agrupacion).size()
        counts_hlp = df_cat[df_cat["dup_host_lic_price"]].groupby(col_agrupacion).size()
        
        tipos_unicos = df_cat[col_agrupacion].drop_duplicates().sort_values(by=col_agrupacion)
        tabla = tipos_unicos.copy()
        
        if len(col_agrupacion) == 1:
            col_key = col_agrupacion[0]
            tabla["Duplicados Mismo Nombre + Anfitrión"] = tabla[col_key].map(counts_nh).fillna(0).astype(int)
            tabla["Duplicados Misma Licencia"] = tabla[col_key].map(counts_lic).fillna(0).astype(int)
            tabla["Duplicados Mismo Nombre + Anfitrión + Licencia"] = tabla[col_key].map(counts_triple).fillna(0).astype(int)
            tabla["Duplicados Anfitrión + Licencia"] = tabla[col_key].map(counts_hl).fillna(0).astype(int)
            tabla["Duplicados Anfitrión + Licencia + Precio"] = tabla[col_key].map(counts_hlp).fillna(0).astype(int)
        else:
            tabla["Duplicados Mismo Nombre + Anfitrión"] = tabla.set_index(col_agrupacion).index.map(counts_nh).fillna(0).astype(int)
            tabla["Duplicados Misma Licencia"] = tabla.set_index(col_agrupacion).index.map(counts_lic).fillna(0).astype(int)
            tabla["Duplicados Mismo Nombre + Anfitrión + Licencia"] = tabla.set_index(col_agrupacion).index.map(counts_triple).fillna(0).astype(int)
            tabla["Duplicados Anfitrión + Licencia"] = tabla.set_index(col_agrupacion).index.map(counts_hl).fillna(0).astype(int)
            tabla["Duplicados Anfitrión + Licencia + Precio"] = tabla.set_index(col_agrupacion).index.map(counts_hlp).fillna(0).astype(int)
            
        tablas_resultado[cat] = tabla
        
    print("================ TABLA 1 (RECALCULADA): APARTAMENTOS TURÍSTICOS (HUTB / TU) ================")
    display(tablas_resultado["Apartamento turístico (HUTB / TU)"])
    
    print("\n================ TABLA 2 (RECALCULADA): SIN LICENCIA / NULA ================")
    display(tablas_resultado["Sin licencia / Nula"])
    
    return df_dedup, tablas_resultado

# Ejecutar desduplicación acumulada y recalcular tablas
df_activos_dedup, tablas_dedup = aplicar_desduplicacion_progresiva(df)


In [ ]:
# 10. Gráfico lineal comparativo de duplicados (incluyendo línea morada para Mismo Precio + Nombre + Anfitrión en Sin Licencia)

import matplotlib.pyplot as plt

def graficar_duplicados_lineal(df_desduplicado):
    """
    Genera los gráficos lineales comparativos para 'Entire home/apt' en Tabla 1 y Tabla 2:
    - Línea Roja: Duplicados por Mismo Nombre + Anfitrión.
    - Línea Azul: Duplicados por Anfitrión + Licencia.
    - Línea Verde: Duplicados por Mismo Nombre, Anfitrión y Licencia.
    - Línea Morada (solo en Tabla 2 Sin Licencia): Duplicados por Mismo Precio, Nombre y Anfitrión.
    
    Args:
        df_desduplicado (pd.DataFrame): DataFrame desduplicado obtenido en la celda 9.
    """
    categorias = ["Apartamento turístico (HUTB / TU)", "Sin licencia / Nula"]
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    for i, cat in enumerate(categorias):
        sub = df_desduplicado[(df_desduplicado["categoria_licencia"] == cat) & (df_desduplicado["room_type"] == "Entire home/apt")].copy()
        
        # 1. Grupos duplicados por Mismo Nombre + Anfitrión (Rojo)
        nh_sizes = sub[sub["dup_nombre_host"]].groupby(["name", "host_id"]).size()
        nh_dist = nh_sizes.value_counts().sort_index()
        
        # 2. Grupos duplicados por Anfitrión + Licencia (Azul)
        hl_sizes = sub[sub["dup_host_lic"]].groupby(["host_id", "license"]).size()
        hl_dist = hl_sizes.value_counts().sort_index()
        
        # 3. Grupos duplicados por Mismo Nombre + Anfitrión + Licencia (Verde)
        triple_sizes = sub[sub["dup_triple"]].groupby(["name", "host_id", "license"]).size()
        triple_dist = triple_sizes.value_counts().sort_index()
        
        # 4. Grupos duplicados por Mismo Precio + Nombre + Anfitrión (Morado)
        pnh_sizes = sub[sub["dup_precio_nombre_host"]].groupby(["price", "name", "host_id"]).size()
        pnh_dist = pnh_sizes.value_counts().sort_index()
        
        max_x = max(
            nh_dist.index.max() if len(nh_dist) > 0 else 2,
            hl_dist.index.max() if len(hl_dist) > 0 else 2,
            triple_dist.index.max() if len(triple_dist) > 0 else 2,
            pnh_dist.index.max() if len(pnh_dist) > 0 else 2
        )
        x_range = list(range(2, max_x + 1))
        
        y_nh = [nh_dist.get(x, 0) for x in x_range]
        y_hl = [hl_dist.get(x, 0) for x in x_range]
        y_triple = [triple_dist.get(x, 0) for x in x_range]
        y_pnh = [pnh_dist.get(x, 0) for x in x_range]
        
        axes[i].plot(x_range, y_nh, color="red", marker="o", linewidth=2, label="Mismo Nombre + Anfitrión")
        axes[i].plot(x_range, y_hl, color="blue", marker="s", linewidth=2, label="Anfitrión + Licencia")
        axes[i].plot(x_range, y_triple, color="green", marker="^", linewidth=2, label="Nombre + Anfitrión + Licencia")
        
        if cat == "Sin licencia / Nula":
            axes[i].plot(x_range, y_pnh, color="purple", marker="d", linewidth=2, label="Precio + Nombre + Anfitrión")
            
        axes[i].set_title(f"Grupos duplicados Entire home/apt - {cat}", fontsize=11)
        axes[i].set_xlabel("Número de anuncios duplicados por grupo")
        axes[i].set_ylabel("Cantidad de grupos duplicados")
        axes[i].legend()
        axes[i].grid(True, linestyle="--", alpha=0.6)
        
    plt.tight_layout()
    plt.show()

# Generar gráfico lineal comparativo actualizado
graficar_duplicados_lineal(df_activos_dedup)


In [ ]:
# 11. Tabla resumen final de duplicados tras todo el proceso de desduplicación (VUTs/TU y Sin Licencia)

def mostrar_tablas_finales_duplicados(df_desduplicado):
    """
    Genera y muestra en una celda dedicada las dos tablas resumen de duplicados finales 
    para 'Apartamento turístico (HUTB / TU)' y 'Sin licencia / Nula' desglosadas por 'room_type'.
    
    Args:
        df_desduplicado (pd.DataFrame): DataFrame limpio tras todo el proceso de desduplicación acumulado.
        
    Returns:
        tuple: (pd.DataFrame tabla_vut, pd.DataFrame tabla_sin_licencia)
    """
    categorias = ["Apartamento turístico (HUTB / TU)", "Sin licencia / Nula"]
    tablas = {}
    
    for cat in categorias:
        df_cat = df_desduplicado[df_desduplicado["categoria_licencia"] == cat]
        
        counts_nh = df_cat[df_cat["dup_nombre_host"]].groupby("room_type").size()
        counts_lic = df_cat[df_cat["dup_licencia"]].groupby("room_type").size()
        counts_triple = df_cat[df_cat["dup_triple"]].groupby("room_type").size()
        counts_hl = df_cat[df_cat["dup_host_lic"]].groupby("room_type").size()
        counts_hlp = df_cat[df_cat["dup_host_lic_price"]].groupby("room_type").size()
        
        tabla = pd.DataFrame({"Tipo de Habitación (room_type)": df_cat["room_type"].unique()}).sort_values(by="Tipo de Habitación (room_type)")
        tabla["Duplicados Mismo Nombre + Anfitrión"] = tabla["Tipo de Habitación (room_type)"].map(counts_nh).fillna(0).astype(int)
        tabla["Duplicados Misma Licencia"] = tabla["Tipo de Habitación (room_type)"].map(counts_lic).fillna(0).astype(int)
        tabla["Duplicados Nombre + Anfitrión + Licencia"] = tabla["Tipo de Habitación (room_type)"].map(counts_triple).fillna(0).astype(int)
        tabla["Duplicados Anfitrión + Licencia"] = tabla["Tipo de Habitación (room_type)"].map(counts_hl).fillna(0).astype(int)
        tabla["Duplicados Anfitrión + Licencia + Precio"] = tabla["Tipo de Habitación (room_type)"].map(counts_hlp).fillna(0).astype(int)
        
        tablas[cat] = tabla
        
    print("================ TABLA FINAL 1: APARTAMENTOS TURÍSTICOS (HUTB / TU - VUTs) ================")
    display(tablas["Apartamento turístico (HUTB / TU)"])
    
    print("\n================ TABLA FINAL 2: SIN LICENCIA / NULA ================")
    display(tablas["Sin licencia / Nula"])
    
    return tablas["Apartamento turístico (HUTB / TU)"], tablas["Sin licencia / Nula"]

# Ejecutar y visualizar las dos tablas resumen finales en esta nueva celda
tabla_vut_final, tabla_sin_final = mostrar_tablas_finales_duplicados(df_activos_dedup)


In [ ]:
# 12. Desduplicación secuencial en Tabla 1 (Roja por GPS y Precio, Azul por Licencia+GPS) y gráficos finales

import matplotlib.pyplot as plt
import pandas as pd

def aplicar_desduplicacion_tabla1(df_desduplicado):
    """
    1. Desduplica Tabla 2 (Sin Licencia) conservando 1 solo registro por anuncio real (Mismo Nombre + Anfitrión).
    2. Desduplica Tabla 1 (Apartamentos Turísticos VUTs) secuencialmente:
       - Línea ROJA por Coordenadas GPS (name, host_id, lat, lon).
       - Línea ROJA por Precio (name, host_id, price).
       - Línea AZUL por Licencia y GPS (license, lat, lon).
    3. Muestra las tablas y el gráfico comparativo de duplicados restantes.
    
    Args:
        df_desduplicado (pd.DataFrame): Dataset desduplicado acumulado de la celda 9.
        
    Returns:
        pd.DataFrame: Dataset final completamente limpio.
    """
    # 1. Limpieza total de Tabla 2 (Sin Licencia / Nula)
    mask_sin = df_desduplicado["categoria_licencia"] == "Sin licencia / Nula"
    df_sin_limpio = df_desduplicado[mask_sin].drop_duplicates(subset=["name", "host_id"], keep="first")
    df_paso1 = pd.concat([df_desduplicado[~mask_sin], df_sin_limpio], ignore_index=True)
    
    # 2. Desduplicación secuencial de Tabla 1 (Apartamentos Turísticos VUTs)
    mask_t1 = df_paso1["categoria_licencia"] == "Apartamento turístico (HUTB / TU)"
    df_t1 = df_paso1[mask_t1].copy()
    
    tot_t1_inicial = len(df_t1)
    
    # Paso A: Desduplicar ROJA por Nombre + Anfitrión + Coordenadas GPS
    df_t1_gps = df_t1.drop_duplicates(subset=["name", "host_id", "latitude", "longitude"], keep="first")
    
    # Paso B: Desduplicar ROJA por Nombre + Anfitrión + Precio
    df_t1_precio = df_t1_gps.drop_duplicates(subset=["name", "host_id", "price"], keep="first")
    
    # Paso C: Desduplicar AZUL por Licencia + Coordenadas GPS
    mask_lic_t1 = df_t1_precio["license"].notna()
    df_t1_lic_gps = df_t1_precio[mask_lic_t1].drop_duplicates(subset=["license", "latitude", "longitude"], keep="first")
    df_t1_dedup = pd.concat([df_t1_lic_gps, df_t1_precio[~mask_lic_t1]], ignore_index=True)
    
    tot_t1_final = len(df_t1_dedup)
    eliminados_t1 = tot_t1_inicial - tot_t1_final
    
    print(f"--- DESDUPLICACIÓN EN TABLA 1 (APARTAMENTOS TURÍSTICOS VUTs) ---")
    print(f"Registros en Tabla 1 iniciales: {tot_t1_inicial:,}")
    print(f"Registros en Tabla 1 tras desduplicar Roja por GPS y Precio, y Azul por Licencia+GPS: {tot_t1_final:,} (se eliminaron {eliminados_t1:,} duplicados sobrantes)\n")
    
    df_limpio_final = pd.concat([df_paso1[~mask_t1], df_t1_dedup], ignore_index=True)
    
    # Recalcular banderas de duplicidad en Tabla 1
    df_t1_dedup["dup_nombre_host"] = df_t1_dedup.duplicated(subset=["name", "host_id"], keep=False)
    df_t1_dedup["dup_licencia"] = df_t1_dedup["license"].notna() & df_t1_dedup.duplicated(subset=["license"], keep=False)
    
    # Subconjuntos restantes
    df_nh = df_t1_dedup[df_t1_dedup["dup_nombre_host"]]
    df_lic = df_t1_dedup[df_t1_dedup["dup_licencia"]]
    
    var_nh = {
        "Mismo Barrio": df_nh.duplicated(subset=["name", "host_id", "neighbourhood"], keep=False).sum(),
        "Mismo Mínimo de Noches": df_nh.duplicated(subset=["name", "host_id", "minimum_nights"], keep=False).sum(),
        "Mismas Coordenadas GPS": df_nh.duplicated(subset=["name", "host_id", "latitude", "longitude"], keep=False).sum(),
        "Mismo Precio": df_nh.duplicated(subset=["name", "host_id", "price"], keep=False).sum()
    }
    
    var_lic = {
        "Mismo Barrio": df_lic.duplicated(subset=["license", "neighbourhood"], keep=False).sum(),
        "Mismo Mínimo de Noches": df_lic.duplicated(subset=["license", "minimum_nights"], keep=False).sum(),
        "Mismas Coordenadas GPS": df_lic.duplicated(subset=["license", "latitude", "longitude"], keep=False).sum(),
        "Mismo Precio": df_lic.duplicated(subset=["license", "price"], keep=False).sum()
    }
    
    # Gráficos
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    s_nh = pd.Series(var_nh).sort_values(ascending=True)
    s_lic = pd.Series(var_lic).sort_values(ascending=True)
    
    tot_nh = len(df_nh)
    tot_lic = len(df_lic)
    
    b1 = axes[0].barh(s_nh.index, s_nh.values, color="crimson", edgecolor="darkred")
    axes[0].set_title(f"Tabla 1: Duplicados Mismo Nombre + Anfitrión ({tot_nh} restantes)\nCoincidencias en otras variables", fontsize=11)
    axes[0].set_xlabel("Número de anuncios duplicados que coinciden")
    axes[0].grid(axis="x", linestyle="--", alpha=0.6)
    for b in b1:
        w = b.get_width()
        axes[0].text(w + 1, b.get_y() + b.get_height()/2, f"{int(w)}", va="center", fontweight="bold")
        
    b2 = axes[1].barh(s_lic.index, s_lic.values, color="dodgerblue", edgecolor="navy")
    axes[1].set_title(f"Tabla 1: Duplicados Misma Licencia ({tot_lic} restantes)\nCoincidencias en otras variables", fontsize=11)
    axes[1].set_xlabel("Número de anuncios duplicados que coinciden")
    axes[1].grid(axis="x", linestyle="--", alpha=0.6)
    for b in b2:
        w = b.get_width()
        axes[1].text(w + 1, b.get_y() + b.get_height()/2, f"{int(w)}", va="center", fontweight="bold")
        
    plt.tight_layout()
    plt.show()
    
    return df_limpio_final

# Ejecutar desduplicaciones secuenciales en Tabla 1 y generar gráficos
df_limpio_final = aplicar_desduplicacion_tabla1(df_activos_dedup)


In [ ]:
# 13. Mapa de Barcelona con límites de barrios (sin nombres), geolocalización (Cuadrados=Pisos, Círculos=Habitaciones) y < 200m

import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def haversine_m(lat1, lon1, lat2, lon2):
    """Calcula la distancia geodésica en metros entre dos coordenadas (Haversine)."""
    R = 6371000  # Radio terrestre en metros
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2.0) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

def geolocalizar_duplicados_200m(df_limpio):
    """
    1. Carga las geometrías de los barrios de Barcelona (GeoJSON).
    2. Identifica los alojs. duplicados de la Tabla 1 (VUTs) y calcula cuáles se ubican a < 200 metros.
    3. Dibuja el mapa completo sin etiquetas de texto de nombres de barrios:
       - Límites perimetrales de los barrios.
       - Cuadrados ('s'): Apartamentos enteros ('Entire home/apt').
       - Círculos ('o'): Habitaciones privadas ('Private room').
       - Color Rojo/Naranja: Distancia < 200 metros.
       - Color Azul/Cian: Distancia >= 200 metros.
    
    Args:
        df_limpio (pd.DataFrame): Dataset limpio tras las desduplicaciones de la celda 12.
    """
    # Cargar GeoJSON de barrios de Barcelona
    ruta_geojson = "data/raw/geometria/insideairbnb_barrios_barcelona.geojson"
    with open(ruta_geojson, "r", encoding="utf-8") as f:
        geo_barrios = json.load(f)
        
    mask_t1 = df_limpio["categoria_licencia"] == "Apartamento turístico (HUTB / TU)"
    df_t1 = df_limpio[mask_t1].copy()
    
    df_t1["dup_nombre_host"] = df_t1.duplicated(subset=["name", "host_id"], keep=False)
    df_t1["dup_licencia"] = df_t1["license"].notna() & df_t1.duplicated(subset=["license"], keep=False)
    
    df_dup = df_t1[df_t1["dup_nombre_host"] | df_t1["dup_licencia"]].copy()
    
    ids_cercanos_200m = set()
    
    # Mismo Nombre + Anfitrión < 200m
    for name, group in df_dup[df_dup["dup_nombre_host"]].groupby(["name", "host_id"]):
        if len(group) > 1:
            lats, lons, ids = group["latitude"].values, group["longitude"].values, group["id"].values
            for i in range(len(lats)):
                for j in range(i + 1, len(lats)):
                    if haversine_m(lats[i], lons[i], lats[j], lons[j]) < 200:
                        ids_cercanos_200m.add(ids[i])
                        ids_cercanos_200m.add(ids[j])
                        
    # Misma Licencia < 200m
    for lic, group in df_dup[df_dup["dup_licencia"]].groupby("license"):
        if len(group) > 1:
            lats, lons, ids = group["latitude"].values, group["longitude"].values, group["id"].values
            for i in range(len(lats)):
                for j in range(i + 1, len(lats)):
                    if haversine_m(lats[i], lons[i], lats[j], lons[j]) < 200:
                        ids_cercanos_200m.add(ids[i])
                        ids_cercanos_200m.add(ids[j])
                        
    df_dup["es_menos_200m"] = df_dup["id"].isin(ids_cercanos_200m)
    
    tot_dup = len(df_dup)
    tot_200 = df_dup["es_menos_200m"].sum()
    pct_200 = (tot_200 / tot_dup * 100) if tot_dup > 0 else 0
    
    print(f"--- ANÁLISIS ESPACIAL DE DUPLICADOS EN BARCELONA (< 200m) ---")
    print(f"Total de anuncios duplicados en Tabla 1 (VUTs): {tot_dup:,}")
    print(f"Anuncios a MENOS DE 200 METROS entre sí: {tot_200:,} ({pct_200:.2f}% del total)\n")
    
    fig, ax = plt.subplots(figsize=(13, 10))
    
    # Dibuja polígonos de barrios sin texto
    for feat in geo_barrios["features"]:
        geom = feat["geometry"]
        if geom["type"] == "Polygon":
            coords_list = [geom["coordinates"]]
        elif geom["type"] == "MultiPolygon":
            coords_list = geom["coordinates"]
        else:
            continue
            
        for poly in coords_list:
            for ring in poly:
                xs = [pt[0] for pt in ring]
                ys = [pt[1] for pt in ring]
                ax.plot(xs, ys, color="slategray", linewidth=0.7, linestyle="--")
                
    # Dibuja puntos de alojamientos con simbología
    e_200 = df_dup[(df_dup["room_type"] == "Entire home/apt") & df_dup["es_menos_200m"]]
    e_far = df_dup[(df_dup["room_type"] == "Entire home/apt") & (~df_dup["es_menos_200m"])]
    p_200 = df_dup[(df_dup["room_type"] == "Private room") & df_dup["es_menos_200m"]]
    p_far = df_dup[(df_dup["room_type"] == "Private room") & (~df_dup["es_menos_200m"])]
    
    ax.scatter(e_200["longitude"], e_200["latitude"], color="crimson", marker="s", s=70, label=f"Piso entero < 200m ({len(e_200)})")
    ax.scatter(e_far["longitude"], e_far["latitude"], color="royalblue", marker="s", s=40, alpha=0.6, label=f"Piso entero >= 200m ({len(e_far)})")
    
    ax.scatter(p_200["longitude"], p_200["latitude"], color="darkorange", marker="o", s=70, label=f"Habitación privada < 200m ({len(p_200)})")
    ax.scatter(p_far["longitude"], p_far["latitude"], color="mediumturquoise", marker="o", s=40, alpha=0.6, label=f"Habitación privada >= 200m ({len(p_far)})")
    
    ax.set_title("Mapa de Barcelona con límites de barrios (sin nombres) y alojs. duplicados (< 200m)\n(Cuadrados = Pisos enteros | Círculos = Habitaciones privadas)", fontsize=11)
    ax.set_xlabel("Longitud")
    ax.set_ylabel("Latitud")
    ax.legend(loc="upper left")
    ax.grid(True, linestyle=":", alpha=0.4)
    
    plt.tight_layout()
    plt.show()
    
    return df_dup

# Ejecutar mapa sin nombres de barrios y a menos de 200m
df_mapa_dup = geolocalizar_duplicados_200m(df_limpio_final)


In [ ]:
# 14. Mapa de Barcelona con límites de barrios (sin nombres), geolocalización (Cuadrados=Pisos, Círculos=Habitaciones) y < 150m

import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def geolocalizar_duplicados_150m(df_limpio):
    """
    1. Carga las geometrías de los barrios de Barcelona (GeoJSON).
    2. Identifica los alojs. duplicados de la Tabla 1 (VUTs) y calcula cuáles se ubican a < 150 metros.
    3. Dibuja el mapa completo sin etiquetas de texto de nombres de barrios:
       - Límites perimetrales de los barrios.
       - Cuadrados ('s'): Apartamentos enteros ('Entire home/apt').
       - Círculos ('o'): Habitaciones privadas ('Private room').
       - Color Rojo/Naranja: Distancia < 150 metros.
       - Color Azul/Cian: Distancia >= 150 metros.
    
    Args:
        df_limpio (pd.DataFrame): Dataset limpio tras las desduplicaciones de la celda 12.
    """
    # Cargar GeoJSON de barrios de Barcelona
    ruta_geojson = "data/raw/geometria/insideairbnb_barrios_barcelona.geojson"
    with open(ruta_geojson, "r", encoding="utf-8") as f:
        geo_barrios = json.load(f)
        
    mask_t1 = df_limpio["categoria_licencia"] == "Apartamento turístico (HUTB / TU)"
    df_t1 = df_limpio[mask_t1].copy()
    
    df_t1["dup_nombre_host"] = df_t1.duplicated(subset=["name", "host_id"], keep=False)
    df_t1["dup_licencia"] = df_t1["license"].notna() & df_t1.duplicated(subset=["license"], keep=False)
    
    df_dup = df_t1[df_t1["dup_nombre_host"] | df_t1["dup_licencia"]].copy()
    
    ids_cercanos_150m = set()
    
    # Mismo Nombre + Anfitrión < 150m
    for name, group in df_dup[df_dup["dup_nombre_host"]].groupby(["name", "host_id"]):
        if len(group) > 1:
            lats, lons, ids = group["latitude"].values, group["longitude"].values, group["id"].values
            for i in range(len(lats)):
                for j in range(i + 1, len(lats)):
                    if haversine_m(lats[i], lons[i], lats[j], lons[j]) < 150:
                        ids_cercanos_150m.add(ids[i])
                        ids_cercanos_150m.add(ids[j])
                        
    # Misma Licencia < 150m
    for lic, group in df_dup[df_dup["dup_licencia"]].groupby("license"):
        if len(group) > 1:
            lats, lons, ids = group["latitude"].values, group["longitude"].values, group["id"].values
            for i in range(len(lats)):
                for j in range(i + 1, len(lats)):
                    if haversine_m(lats[i], lons[i], lats[j], lons[j]) < 150:
                        ids_cercanos_150m.add(ids[i])
                        ids_cercanos_150m.add(ids[j])
                        
    df_dup["es_menos_150m"] = df_dup["id"].isin(ids_cercanos_150m)
    
    tot_dup = len(df_dup)
    tot_150 = df_dup["es_menos_150m"].sum()
    pct_150 = (tot_150 / tot_dup * 100) if tot_dup > 0 else 0
    
    print(f"--- ANÁLISIS ESPACIAL DE DUPLICADOS EN BARCELONA (< 150m) ---")
    print(f"Total de anuncios duplicados en Tabla 1 (VUTs): {tot_dup:,}")
    print(f"Anuncios a MENOS DE 150 METROS entre sí: {tot_150:,} ({pct_150:.2f}% del total)\n")
    
    fig, ax = plt.subplots(figsize=(13, 10))
    
    # Dibuja polígonos de barrios sin texto
    for feat in geo_barrios["features"]:
        geom = feat["geometry"]
        if geom["type"] == "Polygon":
            coords_list = [geom["coordinates"]]
        elif geom["type"] == "MultiPolygon":
            coords_list = geom["coordinates"]
        else:
            continue
            
        for poly in coords_list:
            for ring in poly:
                xs = [pt[0] for pt in ring]
                ys = [pt[1] for pt in ring]
                ax.plot(xs, ys, color="slategray", linewidth=0.7, linestyle="--")
                
    # Dibuja puntos de alojamientos con simbología
    e_150 = df_dup[(df_dup["room_type"] == "Entire home/apt") & df_dup["es_menos_150m"]]
    e_far = df_dup[(df_dup["room_type"] == "Entire home/apt") & (~df_dup["es_menos_150m"])]
    p_150 = df_dup[(df_dup["room_type"] == "Private room") & df_dup["es_menos_150m"]]
    p_far = df_dup[(df_dup["room_type"] == "Private room") & (~df_dup["es_menos_150m"])]
    
    ax.scatter(e_150["longitude"], e_150["latitude"], color="crimson", marker="s", s=70, label=f"Piso entero < 150m ({len(e_150)})")
    ax.scatter(e_far["longitude"], e_far["latitude"], color="royalblue", marker="s", s=40, alpha=0.6, label=f"Piso entero >= 150m ({len(e_far)})")
    
    ax.scatter(p_150["longitude"], p_150["latitude"], color="darkorange", marker="o", s=70, label=f"Habitación privada < 150m ({len(p_150)})")
    ax.scatter(p_far["longitude"], p_far["latitude"], color="mediumturquoise", marker="o", s=40, alpha=0.6, label=f"Habitación privada >= 150m ({len(p_far)})")
    
    ax.set_title("Mapa de Barcelona con límites de barrios (sin nombres) y alojs. duplicados (< 150m)\n(Cuadrados = Pisos enteros | Círculos = Habitaciones privadas)", fontsize=11)
    ax.set_xlabel("Longitud")
    ax.set_ylabel("Latitud")
    ax.legend(loc="upper left")
    ax.grid(True, linestyle=":", alpha=0.4)
    
    plt.tight_layout()
    plt.show()
    
    return df_dup

# Ejecutar mapa a menos de 150m
df_mapa_150m = geolocalizar_duplicados_150m(df_limpio_final)
